In [ ]:
# Load 
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# Add project root to path so we can import from `generation/`
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
torch.cuda.empty_cache()
pipe.to("cuda")
print("Loaded pipe!")

In [ ]:
from LatentPredictionDataset import Latent
def get_latent_lists_from_seeds(seeds: list[int]) -> list[Latent]:
    generators = [torch.Generator(device="cpu").manual_seed(seed) for seed in seeds]
    latents = [
            pipe.prepare_latents(
                batch_size=1,
                num_channels_latents=pipe.unet.config.in_channels,
                height=512,
                width=512,
                dtype=pipe.unet.dtype,
                device="cpu",
                generator=generator,
            )[0] / pipe.scheduler.init_noise_sigma
            for generator in generators
        ]
    return latents

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from LatentPredictionDataset import create_splits, LatentPredictionDataset

full_dataframe = pd.read_csv("../results/baseline_metrics.csv")
print(f"Loaded {len(full_dataframe)} rows")

# 1. Create the split DataFrames
train_df, val_df = create_splits(full_dataframe)
print(f"Train df Rows: {len(train_df)} | Val dfRows: {len(val_df)}")

# 2. Create two separate Dataset instances
# (They don't know about each other, they just see their own data)
train_dataset = LatentPredictionDataset(
    metrics_df=train_df,
    seeds_to_latent=get_latent_lists_from_seeds,
    alpha_range=(0.1, 0.9),
    dtype=torch.float32
)

val_dataset = LatentPredictionDataset(
    metrics_df=val_df,
    seeds_to_latent=get_latent_lists_from_seeds,
    alpha_range=(0.1, 0.9),
    dtype=torch.float32
)

print(f"Train dataset length: {len(train_dataset)} | Val dataset length: {len(val_dataset)}")

# Train latent Predictor

In [ ]:
from training.LatentPredictor import LatentShiftNetwork
latent_predictor = LatentShiftNetwork()
# print(latent_predictor)

# example = train_dataset[1]
# example_prediction = latent_predictor(example["input_winner_latent"], example["input_loser_latents"])

In [ ]:
from training.LatentPredictorTrainer import LatentShiftTrainer, LatentTrainerConfig

BATCH_SIZE = 64

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False) # No shuffle for Val usually")

# 1. Configure hyperparameters
# Start with modest settings. If directionality is low, increase lambda_triplet.
config = LatentTrainerConfig(
    lr=1e-3, 
    epochs=15, 
    triplet_weight=0.0,   # Scale the ranking loss. For now, only MSE on the winner latent
    triplet_margin=1.0,   # Ensure significant separation
    device="cuda"
)

# 2. Initialize Trainer with your existing model and loaders
trainer = LatentShiftTrainer(
    model=latent_predictor, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    config=config
)

# 3. Train
trainer.fit()

# Save the model

In [ ]:
# Define path
# save_path = "../latent_predictors/latent_predictor_lr=1e-4_epochs=15_triplet_weight=0.0_triplet_margin=1.0.pth"
save_path = "../latent_predictors/v2.pth"

# Save only the weights
torch.save(trainer.model.state_dict(), save_path)

print(f"Model weights saved to {save_path}")

In [ ]:
# Load the model back in
latent_predictor2 = LatentShiftNetwork()
latent_predictor2.load_state_dict(torch.load("../latent_predictors/latent_predictor_lr=1e-4_epochs=15_triplet_weight=0.0_triplet_margin=1.0.pth"))
latent_predictor2.eval()


In [ ]:
# try sampling on a train example
train_example = train_dataset[1]
train_example_prompt = train_df["prompt"][train_df["prompt_id"] == train_example["prompt_id"]].unique().item()

In [ ]:
from PIL import Image
def generate_image(prompt: str, latent: Latent) -> Image.Image:
    return pipe(
        prompt=prompt,
        num_images_per_prompt=1,
        latents=latent.unsqueeze(0), # must unsqueeze
        width=512,
        height=512,
        num_inference_steps=1,
        guidance_scale=0.0,
    ).images[0]

In [ ]:
example_prediction = latent_predictor2(example["input_winner_latent"], example["input_loser_latents"])

In [ ]:
print("Input winner latent:")
display(generate_image(train_example_prompt, train_example["input_winner_latent"].to(pipe.device, dtype=pipe.unet.dtype)).resize((100, 100)))

print("Target winner latent:")
display(generate_image(train_example_prompt, train_example["target_winner_latent"].to(pipe.device, dtype=pipe.unet.dtype)).resize((100, 100)))

print("Prediction:")
display(generate_image(train_example_prompt, example_prediction.to(pipe.device, dtype=pipe.unet.dtype)).resize((100, 100)))


In [ ]:
(example_prediction - train_example["input_winner_latent"]).norm()

In [ ]:
example_prediction.norm()